## 1. 환경 설정

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## 2. prepare_context
- 학생 프로필 정보 (목표 대학, 계열)
- 최근 대화 내역 (학생 ↔ 선생님 대화 로그 일부)

두 가지를 받아서 질문 분석(analyze_question) 단계에 넘길 “맥락 포함 질문”을 생성

In [2]:
from typing import TypedDict, List, Dict

# LangGraph 상태 구조 정의
class QAState(TypedDict):
    question: str
    student_profile: Dict[str, str]
    recent_dialogues: List[Dict[str, str]]
    context: str
    category: str
    faq_answer: str
    search_results: List[Dict]
    candidate_answer: str
    evaluation: str
    final_answer: str
    sources: List[str]


In [3]:
def prepare_context(state: QAState) -> QAState:
    """
    학생 프로필과 최근 대화 내역을 종합하여 context 생성
    """
    # 학생 프로필 불러오기
    profile = state.get("student_profile", {})
    target_uni = profile.get("target_university", "미지정")
    track = profile.get("track", "계열 미지정")

    # 최근 대화 내역 가져오기 (학생과 선생님 5개 정도)
    dialogues = state.get("recent_dialogues", [])
    dialogue_summary = " ".join(
        [f"{d['role']}: {d['message']}" for d in dialogues[-5:]]
    )

    # 질문과 맥락 결합
    state["context"] = (
        f"[학생 프로필] 목표 대학: {target_uni}, 계열: {track}\n"
        f"[최근 대화 요약] {dialogue_summary}\n"
        f"[학생 질문] {state['question']}"
    )

    return state


In [4]:
# 초기 상태 정의
init_state: QAState = {
    "question": "혹시 중앙대학교 이과는 어떤 과목을 준비해야 하나요?",
    "student_profile": {
        "target_university": "중앙대학교",
        "track": "이과"
    },
    "recent_dialogues": [
        {"role": "student", "message": "아직 편입 공부를 시작하진 않았습니다."},
        {"role": "teacher", "message": "언제부터 공부 시작할 계획인가요?"},
        {"role": "student", "message": "다음 주부터 시작할 생각인데 정해지면 이야기해드릴게요."},
        {"role": "teacher", "message": "네 알겠습니다."},
        {"role": "student", "message": "혹시 중앙대학교 이과는 어떤 과목을 준비해야 하나요?"}
    ],
    "context": "",
    "category": "",
    "faq_answer": "",
    "search_results": [],
    "candidate_answer": "",
    "evaluation": "",
    "final_answer": "",
    "sources": []
}

# prepare_context 실행
updated_state = prepare_context(init_state)

print("=== 준비된 Context ===")
print(updated_state["context"])


=== 준비된 Context ===
[학생 프로필] 목표 대학: 중앙대학교, 계열: 이과
[최근 대화 요약] student: 아직 편입 공부를 시작하진 않았습니다. teacher: 언제부터 공부 시작할 계획인가요? student: 다음 주부터 시작할 생각인데 정해지면 이야기해드릴게요. teacher: 네 알겠습니다. student: 혹시 중앙대학교 이과는 어떤 과목을 준비해야 하나요?
[학생 질문] 혹시 중앙대학교 이과는 어떤 과목을 준비해야 하나요?


### 도구 1. Guideline_search
- 가이드라인DB 내에서 검색
### 도구 2. web_search
- 웹 검색 

In [ ]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.retrievers import TavilySearchAPIRetriever
from langchain_core.tools import tool
from typing import List
import pandas as pd
import os


# ======================================================
# 1. Ollama Embeddings 초기화
# ======================================================
# - 반드시 터미널에서 아래 명령으로 모델 받아야 함:
#   ollama pull bge-m3
embeddings_model = OllamaEmbeddings(model="bge-m3")


# ======================================================
# 2. Chroma DB 생성 또는 불러오기
# ======================================================
persist_dir = "./chroma_guideline"
collection_name = "guideline_db"

if not os.path.exists(persist_dir):
    print("📌 최초 실행: CSV에서 DB 생성 중...")
    df = pd.read_csv("GuidelineDB.csv", encoding="utf-8")

    # CSV 전체를 Document 리스트로 변환
    documents = [
        Document(
            page_content=row["question"],
            metadata={"answer": row["answer"], "category": row.get("category", "")}
        )
        for _, row in df.iterrows()
    ]

    # 한 번에 DB 생성
    guideline_db = Chroma.from_documents(
        documents=documents,
        embedding=embeddings_model,
        collection_name=collection_name,
        persist_directory=persist_dir
    )
else:
    print("📌 기존 DB 불러오는 중...")
    guideline_db = Chroma(
        collection_name=collection_name,
        persist_directory=persist_dir,
        embedding_function=embeddings_model
    )


# ======================================================
# 3. Re-rank 모델 설정
# ======================================================
rerank_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
cross_reranker = CrossEncoderReranker(model=rerank_model, top_n=5)

guideline_retriever = ContextualCompressionRetriever(
    base_compressor=cross_reranker,
    base_retriever=guideline_db.as_retriever(search_kwargs={"k": 5}),
)


# ======================================================
# 4. GuidelineDB 검색 도구
# ======================================================
@tool
def guideline_search(query: str) -> List[Document]:
    """GuidelineDB에서 검색합니다."""
    docs = guideline_retriever.invoke(query)

    if len(docs) > 0:
        enriched_docs = []
        for d in docs:
            enriched_docs.append(
                Document(
                    page_content=f"Q: {d.page_content}\nA: {d.metadata['answer']}",
                    metadata={"source": "guidelineDB"}
                )
            )
        return enriched_docs
    
    return [Document(page_content="관련 정보를 찾을 수 없습니다.")]


# ======================================================
# 5. 웹 검색 도구
# ======================================================
web_retriever = ContextualCompressionRetriever(
    base_compressor=cross_reranker,
    base_retriever=TavilySearchAPIRetriever(k=10),
)

@tool
def web_search(query: str) -> List[Document]:
    """데이터베이스에 없는 정보 또는 최신 정보를 웹에서 검색합니다."""
    docs = web_retriever.invoke(query)

    formatted_docs = []
    for doc in docs:
        formatted_docs.append(
            Document(
                page_content=f'<Document href="{doc.metadata["source"]}"/>\n{doc.page_content}\n</Document>',
                metadata={"source": "web search", "url": doc.metadata["source"]}
            )
        )

    if len(formatted_docs) > 0:
        return formatted_docs
    
    return [Document(page_content="관련 정보를 찾을 수 없습니다.")]


C:\Users\이성재\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📌 최초 실행: CSV에서 DB 생성 중...


In [6]:
# --------------------------
# 6. 테스트 실행
# --------------------------
if __name__ == "__main__":
    query = "다변수미적분 진도는 언제 시작해야 하나요?"
    results = guideline_search.invoke(query)

    print("\n=== 검색 결과 ===")
    for r in results:
        print(r.page_content)



=== 검색 결과 ===
Q: 다변수미적분 진도는 언제 시작해야 하나요?
A: 3, 4월에는 선형대수 중심으로, 5월부터 다변수미적분 강의를 시작하는 것이 정규 커리큘럼에 맞는 진행입니다.
Q: 다변수미적분은 언제부터 시작하는 게 좋을까요?
A: 기존 복습이 끝나는 시점부터 바로 시작하시고, 진도는 빠르게 나가되 복습을 병행하세요.


In [9]:
# 도구 목록을 정의 
tools = [guideline_search, web_search]

### LLM 모델 설정 

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import Tool

# .env 파일에서 GOOGLE_API_KEY 불러오기
load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY")

# 기본 LLM - Gemini 사용
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=google_api_key,
    temperature=0,
    streaming=True
)

# 도구 바인딩
llm_with_tools = llm.bind_tools(tools)


In [ ]:
from pprint import pprint

# 도구 호출이 필요한 경우 ai_msg.content가 ''로 나옴
query = "다변수미적분 진도는 언제 시작해야 하나요?"
ai_msg = llm_with_tools.invoke(query)

pprint(ai_msg)
print("-" * 100)

pprint(ai_msg.content)
print("-" * 100)

pprint(ai_msg.tool_calls)
print("-" * 100)

AIMessage(content='', additional_kwargs={'function_call': {'name': 'guideline_search', 'arguments': '{"query": "\\ub2e4\\ubcc0\\uc218\\ubbf8\\uc801\\ubd84 \\uc9c4\\ub3c4\\ub294 \\uc5b8\\uc81c \\uc2dc\\uc791\\ud574\\uc57c \\ud558\\ub098\\uc694?"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--8ef5c97e-73b0-4f44-854e-753df8ba04ca-0', tool_calls=[{'name': 'guideline_search', 'args': {'query': '다변수미적분 진도는 언제 시작해야 하나요?'}, 'id': '1202deac-f627-4775-825e-ee63b1bd055a', 'type': 'tool_call'}], usage_metadata={'input_tokens': 105, 'output_tokens': 29, 'total_tokens': 253, 'input_token_details': {'cache_read': 0}})
----------------------------------------------------------------------------------------------------
''
----------------------------------------------------------------------------------------------------
[{'args': {'query': '다변수미적분 진도는 언제 시작해야 하나요?'},
  'id': '1202deac-f627-4775-825e-ee63b1b